# CNN multiclasses — variante issue de E2 (DA seule)

**Objectif :** comparer un CNN entraîné sur 4 classes au VGG16 multiclasses M4 (et M7 si disponible). On reprend l'architecture convolutionnelle E0/E2/E3 ; seule la tête de classification passe à 4 sorties `softmax`.

**Avant de lancer :** adapter `DATA_DIR` (CSV) et `DATASET_ROOT` (images) à Colab/Drive. Exécuter dans l'ordre. **Le test n'est jamais évalué ici.** Les CSV doivent contenir `filepath`, `target` (binaire) et `class` (4 classes), comme dans M4.

**Hypothèse E2 :** DA seule sur le train, sans oversampling. Ce notebook entraîne **un nouveau modèle multiclasses**, il ne réutilise pas les poids binaires de E2.

Les points de vigilance sont marqués `# CONFLIT` pour faciliter la comparaison avec les notebooks binaires.

In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, CSVLogger
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

# CONFLIT — MODIFICATION : une racine unique pour CSV et images.
# Dans Colab, monter Drive au préalable :
# from google.colab import drive
# drive.mount('/content/drive')
# CONFLIT — MODIFICATION : un seul dossier Drive imposé pour les CSV et images.
# Monter Google Drive dans Colab AVANT cette cellule :
# from google.colab import drive
# drive.mount("/content/drive")
# Modifier uniquement PROJECT_DIR selon le dossier partagé.
PROJECT_DIR = Path("/content/drive/MyDrive/projet_covid")
DATA_DIR = PROJECT_DIR
DATASET_ROOT = PROJECT_DIR / "COVID-19_Radiography_Dataset"
if not DATA_DIR.is_dir():
    raise FileNotFoundError(f"Dossier projet introuvable : {DATA_DIR}. Monter Drive et vérifier PROJECT_DIR.")
if not DATASET_ROOT.is_dir():
    raise FileNotFoundError(f"Dossier images introuvable : {DATASET_ROOT}. Respecter l'arborescence Kaggle.")
OUTPUT_DIR = Path(".")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CLASSES = ["COVID", "Normal", "Lung_Opacity", "Viral Pneumonia"]
EXPECTED = {"train": 13545, "validation": 3387, "test": 4233}
BATCH_SIZE = 32
IMG_HEIGHT, IMG_WIDTH = 224, 224
SEED = 42
tf.keras.utils.set_random_seed(SEED)

train_df = pd.read_csv(DATA_DIR / "train.csv")
val_df = pd.read_csv(DATA_DIR / "validation.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")

# CONFLIT — AJOUT : contrôle du split commun et des étiquettes multiclasse.
for name, frame in (("train", train_df), ("validation", val_df), ("test", test_df)):
    assert {"filepath", "target", "class"}.issubset(frame.columns), f"{name}: colonnes manquantes"
    assert len(frame) == EXPECTED[name], f"{name}: {len(frame)} au lieu de {EXPECTED[name]}"
    assert frame[["filepath", "target", "class"]].notna().all().all(), f"{name}: valeurs nulles"
    assert frame["filepath"].is_unique, f"{name}: images dupliquées"
    assert set(frame["class"].unique()) <= set(CLASSES), f"{name}: classes inconnues"
    assert set(frame["target"].astype(str).unique()) <= {"0", "1"}
    assert (frame["target"].astype(str).eq("1") == frame["class"].eq("COVID")).all(), f"{name}: target / class incohérents"
    print(f"{name}: {len(frame)} images ; classes = {frame['class'].value_counts().to_dict()}")
assert set(train_df["class"]) == set(CLASSES)
assert set(val_df["class"]) == set(CLASSES)
train_paths = set(train_df["filepath"].astype(str))
val_paths = set(val_df["filepath"].astype(str))
test_paths = set(test_df["filepath"].astype(str))
assert not (train_paths & val_paths or train_paths & test_paths or val_paths & test_paths), "Fuite entre splits"

# CONFLIT — MODIFICATION : affectations simples et séparées, comme dans E2/E3.
train_gen_df = train_df.copy()
val_gen_df = val_df.copy()
test_gen_df = test_df.copy()

# CONFLIT — AJOUT : conversion des chemins Windows relatifs vers la racine réelle.
def resolve_image_path(csv_path):
    normalized = str(csv_path).replace("\\", "/")
    marker = "COVID-19_Radiography_Dataset/"
    if marker not in normalized:
        raise ValueError(f"Chemin CSV sans dossier dataset : {csv_path}")
    relative = normalized.split(marker, 1)[1]
    parts = Path(relative).parts
    if len(parts) != 3 or parts[1] != "images" or parts[0] not in CLASSES:
        raise ValueError(f"Structure d'image inattendue : {csv_path}")
    return str(DATASET_ROOT.joinpath(*parts))

train_gen_df["filepath"] = train_gen_df["filepath"].map(resolve_image_path)
val_gen_df["filepath"] = val_gen_df["filepath"].map(resolve_image_path)
test_gen_df["filepath"] = test_gen_df["filepath"].map(resolve_image_path)

# CONFLIT — AJOUT : vérification avant le fit, y compris pour le test non exploité.
for name, frame in (("train", train_gen_df), ("validation", val_gen_df), ("test", test_gen_df)):
    missing = [p for p in frame["filepath"] if not Path(p).is_file()]
    if missing:
        raise FileNotFoundError(f"{name}: {len(missing)} fichiers absents. Exemple : {missing[0]}. Vérifier PROJECT_DIR.")
    print(f"{name}: {len(frame)} chemins existants")


In [ ]:
# CONFLIT — MODIFICATION : variante E2 = DA seule ; aucun oversampling.
train_fit_df = train_gen_df.copy()
assert len(train_fit_df) == EXPECTED["train"]
print("Train sans OS :", train_fit_df["class"].value_counts().to_dict())


In [ ]:
# CONFLIT — MODIFICATION : même DA médicale que E2/E3, sans flip.
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=False
)
val_datagen = ImageDataGenerator(rescale=1./255)

# CONFLIT — MODIFICATION : 4 classes (colonne class) au lieu du target binaire.
train_generator = train_datagen.flow_from_dataframe(
    dataframe=train_fit_df, x_col="filepath", y_col="class",
    target_size=(IMG_HEIGHT, IMG_WIDTH), color_mode="grayscale",
    classes=CLASSES, class_mode="categorical", batch_size=BATCH_SIZE,
    shuffle=True, seed=SEED
)
val_generator = val_datagen.flow_from_dataframe(
    dataframe=val_gen_df, x_col="filepath", y_col="class",
    target_size=(IMG_HEIGHT, IMG_WIDTH), color_mode="grayscale",
    classes=CLASSES, class_mode="categorical", batch_size=BATCH_SIZE,
    shuffle=False
)
assert train_generator.n == len(train_fit_df), "Train : images écartées par le générateur"
assert val_generator.n == EXPECTED["validation"], "Validation : images écartées"
assert val_generator.class_indices == {name: i for i, name in enumerate(CLASSES)}
assert val_generator.shuffle is False
print("Train batches:", len(train_generator), "Val batches:", len(val_generator))
print("Index des classes:", val_generator.class_indices)


In [ ]:
# CONFLIT — MODIFICATION : architecture E0/E2/E3, sortie 4 classes softmax.
# Ne pas charger le modèle binaire : on entraîne un nouveau CNN multiclasses.
def build_cnn_multiclass():
    model = models.Sequential([
        layers.Input(shape=(224, 224, 1)),
        layers.Conv2D(32, (3, 3), activation="relu"),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation="relu"),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(128, (3, 3), activation="relu"),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.5),
        layers.Dense(4, activation="softmax")
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

model = build_cnn_multiclass()
model.summary()


In [ ]:
# CONFLIT — MODIFICATION : entraînement contrôlé, meilleur modèle conservé.
MODEL_PATH = OUTPUT_DIR / "M_CNN_from_E2_multiclass_best.keras"
HISTORY_PATH = OUTPUT_DIR / "M_CNN_from_E2_multiclass_history.csv"
callbacks = [
    EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True),
    ModelCheckpoint(str(MODEL_PATH), monitor="val_loss", save_best_only=True),
    CSVLogger(str(HISTORY_PATH))
]
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=5,
    callbacks=callbacks
)
print("Meilleur checkpoint :", MODEL_PATH)


In [ ]:
# CONFLIT — AJOUT : évaluation du MEILLEUR checkpoint sur validation commune.
# NE PAS évaluer le test ici ; il sera réservé à l'évaluation finale du modèle sélectionné.
best_model = tf.keras.models.load_model(str(MODEL_PATH))
val_generator.reset()
proba = best_model.predict(val_generator, verbose=1)
assert len(proba) == val_generator.n == EXPECTED["validation"]
y_pred = np.argmax(proba, axis=1)
y_true = val_generator.classes
labels = list(range(len(CLASSES)))
report = classification_report(
    y_true, y_pred, labels=labels, target_names=CLASSES,
    output_dict=True, zero_division=0
)
report_df = pd.DataFrame(report).transpose()
report_df.to_csv(OUTPUT_DIR / "M_CNN_from_E2_classification_report.csv")
print(report_df.to_string())
print("Accuracy :", report["accuracy"])
print("F1 macro :", report["macro avg"]["f1-score"])
print("Recall macro :", report["macro avg"]["recall"])
print("Recall COVID :", report["COVID"]["recall"])
print("F1 COVID :", report["COVID"]["f1-score"])

cm = confusion_matrix(y_true, y_pred, labels=labels)
pd.DataFrame(cm, index=CLASSES, columns=CLASSES).to_csv(
    OUTPUT_DIR / "M_CNN_from_E2_confusion_matrix.csv"
)
fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASSES).plot(
    ax=ax, cmap="Blues", xticks_rotation=35, colorbar=False
)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "M_CNN_from_E2_confusion_matrix.png", dpi=160)
plt.show()


In [ ]:
# CONFLIT — AJOUT : courbes depuis le CSV réel (fonctionne aussi après redémarrage).
history_df = pd.read_csv(HISTORY_PATH)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, metric in zip(axes, ["loss", "accuracy"]):
    ax.plot(history_df[metric], label="Train")
    ax.plot(history_df["val_" + metric], label="Validation")
    ax.set_title(metric)
    ax.set_xlabel("Epoch")
    ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "M_CNN_from_E2_learning_curves.png", dpi=160)
plt.show()


## Contrôle avant partage

- Confirmer `train_generator.n` et `val_generator.n == 3387`.
- Vérifier les quatre classes et leur ordre dans `class_indices`.
- Transmettre le `.keras`, le CSV d'historique, le rapport de classification et les figures.
- **Ne pas exécuter d'évaluation sur `test.csv`** avant la sélection définitive du modèle.

**Limite de comparaison :** M4/M7 utilisent VGG16 avec images RGB et `preprocess_input`, ce CNN utilise des images en niveaux de gris avec `rescale=1./255`. C'est intentionnel : on compare deux familles de modèles sur les **mêmes images** et le **même split**, mais avec leurs prétraitements adaptés.